Team coursework case study: credit application risk modelling with PyTorch. The referenced input file is not included; results are not independently reproduced in this portfolio copy.


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv('data/early_2012_2013_loan_sample_with_outcome.csv')

print(f'Dataset shape: {df.shape}')
print(f'Default rate: {df["loan_is_bad"].mean():.1%}  '
      f'({df["loan_is_bad"].sum():,} defaults / {len(df):,} total)')

In [ ]:
# ── Reproduce Q2 cluster labels ────────────────────────────────────────────
# Exactly mirrors the R pipeline: same 5 features, log-transform, Mahalanobis
# outlier removal, PCA (2 PCs), K-Means k=3, seed=55.

cluster_vars = ['int_rate', 'dti', 'annual_inc', 'delinq_2yrs', 'inq_last_6mths']
cluster_raw = df[cluster_vars].dropna().copy()
cluster_raw['annual_inc'] = np.log1p(cluster_raw['annual_inc'])

# Standardise
scaler_c = StandardScaler()
cluster_scaled = scaler_c.fit_transform(cluster_raw)

# Mahalanobis outlier removal (chi-sq, p < 0.001)
from scipy.spatial.distance import mahalanobis
from scipy.stats import chi2
cov_inv = np.linalg.inv(np.cov(cluster_scaled.T))
center  = cluster_scaled.mean(axis=0)
maha    = np.array([mahalanobis(row, center, cov_inv) ** 2
                    for row in cluster_scaled])
maha_pval = 1 - chi2.cdf(maha, df=5)
keep = maha_pval >= 0.001
print(f'Outliers removed: {(~keep).sum()} ({(~keep).mean():.2%})')
print(f'Observations retained: {keep.sum():,}')

cluster_scaled_clean = cluster_scaled[keep]
cluster_idx          = cluster_raw.index[keep]   # original df indices

# PCA — retain 2 PCs (eigenvalue > 1, matching R)
pca = PCA(n_components=2, random_state=SEED)
pc_scores = pca.fit_transform(cluster_scaled_clean)
print(f'Variance explained by PC1+PC2: {pca.explained_variance_ratio_.sum():.1%}')

# K-Means k=3, seed=55 (matching R set.seed(55))
km = KMeans(n_clusters=3, n_init=25, random_state=55)
km_labels = km.fit_predict(pc_scores)

# Map numeric cluster → named label using mean int_rate (same logic as R)
label_map_df = pd.DataFrame({
    'idx':     cluster_idx,
    'cluster': km_labels,
    'int_rate': df.loc[cluster_idx, 'int_rate'].values
})
cluster_mean_ir = label_map_df.groupby('cluster')['int_rate'].mean().sort_values()
label_names = {
    cluster_mean_ir.index[0]: 'Low Risk (Prime)',
    cluster_mean_ir.index[1]: 'Medium Risk (Near-Prime)',
    cluster_mean_ir.index[2]: 'High Risk (Subprime)'
}
label_map_df['Label'] = label_map_df['cluster'].map(label_names)

# Attach labels back to main df (rows not in cluster_idx get NaN → drop later)
df = df.loc[cluster_idx].copy()
df['cluster_label'] = label_map_df.set_index('idx')['Label']

print('\nCluster distribution:')
print(df['cluster_label'].value_counts())

In [ ]:
# ── Feature engineering ────────────────────────────────────────────────────
NUMERIC_FEATURES = [
    'int_rate', 'dti', 'annual_inc', 'delinq_2yrs',
    'inq_last_6mths', 'loan_amnt', 'open_acc'
]
CAT_FEATURES = ['grade', 'home_ownership', 'purpose']
TARGET = 'loan_is_bad'

df_model = df[NUMERIC_FEATURES + CAT_FEATURES + [TARGET, 'cluster_label']].copy()

# Log-transform skewed income
df_model['annual_inc'] = np.log1p(df_model['annual_inc'])

# Encode categoricals with LabelEncoder
for col in CAT_FEATURES:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))

# Drop rows with any remaining NaN in features
df_model = df_model.dropna(subset=NUMERIC_FEATURES + CAT_FEATURES)
print(f'Final modelling dataset: {df_model.shape[0]:,} rows')

X = df_model[NUMERIC_FEATURES + CAT_FEATURES].values.astype(np.float32)
y = df_model[TARGET].astype(int).values
cluster_labels = df_model['cluster_label'].values

INPUT_DIM = X.shape[1]
print(f'Input features: {INPUT_DIM}')
print(f'Class balance — 0: {(y==0).sum():,}  1: {(y==1).sum():,}  '
      f'(ratio {(y==0).sum()/(y==1).sum():.1f}:1)')

In [ ]:
# ── Three-way split: Train 70% / Val 15% / Test 15% ────────────────────────
# Stratify on target to preserve class ratio in all splits

# Step 1: hold out test (15%)
X_temp, X_test, y_temp, y_test, cl_temp, cl_test = train_test_split(
    X, y, cluster_labels,
    test_size=0.15, random_state=SEED, stratify=y
)

# Step 2: split remainder into train (70%) and val (15%)
# 0.15 / 0.85 ≈ 0.1765
X_train, X_val, y_train, y_val, cl_train, cl_val = train_test_split(
    X_temp, y_temp, cl_temp,
    test_size=0.1765, random_state=SEED, stratify=y_temp
)

print(f'Train : {X_train.shape[0]:,} rows  ({X_train.shape[0]/len(X):.1%})')
print(f'Val   : {X_val.shape[0]:,} rows  ({X_val.shape[0]/len(X):.1%})')
print(f'Test  : {X_test.shape[0]:,} rows  ({X_test.shape[0]/len(X):.1%})')

# Scale on train only → apply to val and test (prevent leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# Convert to tensors
def to_tensors(X, y):
    return (torch.tensor(X, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32))

Xt, yt         = to_tensors(X_train, y_train)
Xv, yv         = to_tensors(X_val,   y_val)
Xte, yte       = to_tensors(X_test,  y_test)

# DataLoaders
BATCH = 256
train_loader = DataLoader(TensorDataset(Xt, yt),   batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xv, yv),   batch_size=BATCH, shuffle=False)
test_loader  = DataLoader(TensorDataset(Xte, yte), batch_size=BATCH, shuffle=False)

# Class imbalance weight for loss function
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()],
                           dtype=torch.float32).to(device)
print(f'\npos_weight for BCEWithLogitsLoss: {pos_weight.item():.2f}')

In [ ]:
# ── Model definitions ──────────────────────────────────────────────────────

class BaselineDNN(nn.Module):
    """Simple 2-hidden-layer DNN (mirrors 6.2 tutorial)."""
    def __init__(self, input_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)   # raw logit — BCEWithLogitsLoss handles sigmoid
        )
    def forward(self, x):
        return self.layers(x)


class OptimisedDNN(nn.Module):
    """Advanced DNN with BatchNorm and Dropout (mirrors 7.1 tutorial)."""
    def __init__(self, input_dim, dropout_rate=0.3):
        super().__init__()
        self.network = nn.Sequential(
            # Layer 1
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            # Layer 2
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            # Layer 3
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            # Output
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.network(x)


print('BaselineDNN:')
print(BaselineDNN(INPUT_DIM))
print('\nOptimisedDNN:')
print(OptimisedDNN(INPUT_DIM))

In [ ]:
# ── Generic training function ──────────────────────────────────────────────

def train_model(model, train_loader, val_loader, pos_weight,
                optimiser_type='sgd', lr=0.01, epochs=100,
                patience=10, use_scheduler=False, model_name='model'):
    """
    Trains a binary classification DNN with optional early stopping
    and learning rate scheduling.
    Returns: trained model, train_losses, val_losses
    """
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    if optimiser_type == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    else:  # adamw
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    scheduler = None
    if use_scheduler:
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5
        )

    best_val_loss    = float('inf')
    epochs_no_improve = 0
    best_weights     = None
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        # --- Training ---
        model.train()
        running_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device).unsqueeze(1)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
        epoch_train_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_train_loss)

        # --- Validation ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device).unsqueeze(1)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)
        epoch_val_loss = val_loss / len(val_loader.dataset)
        val_losses.append(epoch_val_loss)

        if use_scheduler and scheduler:
            scheduler.step(epoch_val_loss)

        # Print progress every 10 epochs
        if (epoch + 1) % 10 == 0:
            print(f'[{model_name}] Epoch {epoch+1:3d}/{epochs}  '
                  f'Train Loss: {epoch_train_loss:.4f}  '
                  f'Val Loss: {epoch_val_loss:.4f}')

        # --- Early stopping ---
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_weights  = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if patience and epochs_no_improve >= patience:
                print(f'[{model_name}] Early stopping at epoch {epoch+1} '
                      f'(no improvement for {patience} epochs)')
                break

    if best_weights:
        model.load_state_dict(best_weights)
    return model, train_losses, val_losses

In [ ]:
# ── Train Baseline DNN ─────────────────────────────────────────────────────
baseline = BaselineDNN(INPUT_DIM)

baseline, b_train_losses, b_val_losses = train_model(
    model          = baseline,
    train_loader   = train_loader,
    val_loader     = val_loader,
    pos_weight     = pos_weight,
    optimiser_type = 'sgd',
    lr             = 0.01,
    epochs         = 100,
    patience       = None,   # no early stopping for baseline
    use_scheduler  = False,
    model_name     = 'Baseline'
)

In [ ]:
# ── Train Optimised DNN ────────────────────────────────────────────────────
optimised = OptimisedDNN(INPUT_DIM, dropout_rate=0.3)

optimised, o_train_losses, o_val_losses = train_model(
    model          = optimised,
    train_loader   = train_loader,
    val_loader     = val_loader,
    pos_weight     = pos_weight,
    optimiser_type = 'adamw',
    lr             = 0.001,
    epochs         = 100,
    patience       = 10,
    use_scheduler  = True,
    model_name     = 'Optimised'
)

In [ ]:
# ── Learning Curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, train_l, val_l, title in zip(
    axes,
    [b_train_losses, o_train_losses],
    [b_val_losses,   o_val_losses],
    ['Baseline DNN — Learning Curve',
     'Optimised DNN — Learning Curve']
):
    ax.plot(train_l, label='Training Loss', color='steelblue')
    ax.plot(val_l,   label='Validation Loss', color='orangered', linestyle='--')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('BCE Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('q3_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: q3_learning_curves.png')

---
## Q3c — Model Evaluation

We evaluate using:
- **AUC-ROC**: discrimination ability across all thresholds — most robust for imbalanced data
- **F1 Score**: harmonic mean of Precision and Recall — balances false positives and false negatives
- **Recall (Sensitivity)**: proportion of actual defaults caught — critical for the business (missing a default is costly)
- **Precision**: proportion of flagged loans that truly default

Classification threshold = **0.5** (default); adjust for business trade-offs.


In [ ]:
# ── Prediction helper ──────────────────────────────────────────────────────

def get_predictions(model, loader, threshold=0.5):
    """Returns (probabilities, binary predictions, true labels) as numpy arrays."""
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(device)
            logits = model(inputs).squeeze(1)
            probs  = torch.sigmoid(logits).cpu().numpy()
            preds  = (probs >= threshold).astype(int)
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(targets.numpy())
    return np.array(all_probs), np.array(all_preds), np.array(all_labels)


def print_metrics(name, y_true, y_pred, y_prob):
    """Print a formatted metrics table."""
    auc  = roc_auc_score(y_true, y_prob)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    print(f'  {name:<30}  AUC: {auc:.4f}  F1: {f1:.4f}  '
          f'Precision: {prec:.4f}  Recall: {rec:.4f}')
    return dict(Model=name, AUC=auc, F1=f1, Precision=prec, Recall=rec)


# Overall test-set performance
b_prob, b_pred, b_true = get_predictions(baseline,  test_loader)
o_prob, o_pred, o_true = get_predictions(optimised, test_loader)

print('=' * 75)
print('OVERALL TEST-SET PERFORMANCE')
print('=' * 75)
results_overall = []
results_overall.append(print_metrics('Baseline DNN',  b_true, b_pred, b_prob))
results_overall.append(print_metrics('Optimised DNN', o_true, o_pred, o_prob))
print('=' * 75)

In [ ]:
# ── Confusion Matrices ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_true, y_pred, title in zip(
    axes,
    [b_true, o_true],
    [b_pred, o_pred],
    ['Baseline DNN — Confusion Matrix',
     'Optimised DNN — Confusion Matrix']
):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Not Default', 'Default'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('q3_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: q3_confusion_matrices.png')

In [ ]:
# ── Performance by Q2 Cluster ──────────────────────────────────────────────
# cl_test holds the cluster label for each test observation

CLUSTER_ORDER = ['Low Risk (Prime)', 'Medium Risk (Near-Prime)', 'High Risk (Subprime)']

print('=' * 75)
print('TEST-SET PERFORMANCE BY Q2 BORROWER CLUSTER')
print('=' * 75)

cluster_results = []

for cluster in CLUSTER_ORDER:
    mask = (cl_test == cluster)
    n    = mask.sum()
    if n < 10:
        continue
    print(f'\n  Cluster: {cluster}  (n={n:,})')
    r_b = print_metrics('  Baseline DNN',
                        b_true[mask], b_pred[mask], b_prob[mask])
    r_o = print_metrics('  Optimised DNN',
                        o_true[mask], o_pred[mask], o_prob[mask])
    r_b['Cluster'] = cluster
    r_o['Cluster'] = cluster
    cluster_results.extend([r_b, r_o])

print('=' * 75)

cluster_df = pd.DataFrame(cluster_results)
print('\nFull cluster x model results table:')
print(cluster_df[['Cluster','Model','AUC','F1','Precision','Recall']]
      .to_string(index=False))

In [ ]:
# ── Visualisation: AUC by Cluster ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

metrics_to_plot = ['AUC', 'Recall']
titles          = ['AUC-ROC by Cluster', 'Recall by Cluster']
ylims           = [(0.4, 1.0), (0.0, 1.0)]

colors = {'Baseline DNN': '#1565C0', 'Optimised DNN': '#E65100'}
x      = np.arange(len(CLUSTER_ORDER))
width  = 0.35

for ax, metric, title, ylim in zip(axes, metrics_to_plot, titles, ylims):
    for i, model_name in enumerate(['Baseline DNN', 'Optimised DNN']):
        vals = [
            cluster_df.loc[
                (cluster_df['Cluster'] == c) & (cluster_df['Model'].str.strip() == model_name),
                metric
            ].values[0]
            if len(cluster_df.loc[
                (cluster_df['Cluster'] == c) & (cluster_df['Model'].str.strip() == model_name)
            ]) > 0 else 0
            for c in CLUSTER_ORDER
        ]
        offset = (i - 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=model_name,
                      color=colors[model_name], alpha=0.85)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=8)

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(['Low Risk\n(Prime)', 'Medium Risk\n(Near-Prime)',
                         'High Risk\n(Subprime)'], fontsize=9)
    ax.set_ylim(ylim)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('q3_cluster_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: q3_cluster_performance.png')

In [ ]:
# ── Summary Table: Overall + By Cluster ───────────────────────────────────
# Combine overall and cluster results into one clean summary table

overall_df = pd.DataFrame(results_overall)
overall_df['Cluster'] = 'Overall'
overall_df['Model']   = overall_df['Model'].str.strip()

cluster_df['Model'] = cluster_df['Model'].str.strip()

summary = pd.concat([overall_df, cluster_df], ignore_index=True)
summary = summary[['Cluster', 'Model', 'AUC', 'F1', 'Precision', 'Recall']]
for col in ['AUC', 'F1', 'Precision', 'Recall']:
    summary[col] = summary[col].round(4)

print('\n=== COMPLETE RESULTS SUMMARY ===')
print(summary.to_string(index=False))